# Khối B: Chia dữ liệu

**Đầu vào:** `meta.csv` từ Khối A (mỗi dòng là một ảnh, có mã bệnh nhân, nhóm `train_val` hoặc `test`, đường dẫn ảnh và 14 cột nhãn 0/1).

**Đầu ra:** ba file `train.csv`, `val.csv`, `test.csv` lưu trên Google Drive.

| Mục | Nội dung |
| --- | --- |
| Nhận vào | Bảng thông tin ảnh từ Khối A |
| Làm gì | Chia ảnh thành ba nhóm: học (train), thử nghiệm trong lúc làm (validation), và chấm điểm cuối cùng (test), theo đúng bệnh nhân, không theo ảnh (giải thích ở mục 2) |
| Tạo ra | Ba danh sách ảnh cho ba nhóm trên |
| Vì sao cần | Không được dùng cùng dữ liệu vừa để dạy mô hình vừa để chấm điểm mô hình, nếu không kết quả sẽ không đáng tin |

**Quy tắc chia**

1. Nhóm test là toàn bộ ảnh có `official = test` (danh sách chính thức của NIH), giữ nguyên.
2. Phần `train_val` được chia theo bệnh nhân bằng `GroupShuffleSplit`; 12,5% số bệnh nhân làm validation.
3. Thử 200 cách chia ngẫu nhiên. Loại cách nào có bệnh nào trong validation ít hơn 10 ảnh dương tính. Trong các cách còn lại, chọn cách có tỉ lệ mỗi bệnh giữa train và validation gần nhau nhất.
4. Kiểm tra không có bệnh nhân nào nằm ở hai nhóm khác nhau.

**Lưu ý:** notebook này không cần tải ảnh, chỉ cần `meta.csv` đã lưu trên Drive từ Khối A.

## 1. Nối Drive và đọc meta.csv

In [ ]:
import os

import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

META_FILE = '/content/drive/MyDrive/CXR_Project/data/meta.csv'
OUT_DIR = '/content/drive/MyDrive/CXR_Project/data/'

assert os.path.exists(META_FILE), 'Không thấy meta.csv: hãy chạy Khối A và lưu meta.csv lên Drive trước'

df = pd.read_csv(META_FILE)
print('Số ảnh:', len(df), '| Số bệnh nhân:', df['patient'].nunique())
print(df['official'].value_counts().to_string())

## 2. Vì sao chia theo bệnh nhân, không chia theo ảnh

Một bệnh nhân có thể có nhiều ảnh chụp ở nhiều thời điểm. Nếu chia theo ảnh, ảnh của cùng một người sẽ rơi vào cả nhóm học lẫn nhóm thử nghiệm. Mô hình có thể học cách nhận ra người đó thay vì nhận ra bệnh, và điểm số sẽ đẹp một cách giả tạo. Chia theo bệnh nhân bảo đảm toàn bộ ảnh của một người nằm trọn trong một nhóm.

In [ ]:
imgs_per_patient = df.groupby('patient').size()
print('Trung bình số ảnh mỗi bệnh nhân:', round(imgs_per_patient.mean(), 2))
print('Bệnh nhân có từ 2 ảnh trở lên:', int((imgs_per_patient > 1).sum()), 'trên', len(imgs_per_patient))
print('Nhiều ảnh nhất của một bệnh nhân:', int(imgs_per_patient.max()))

## 3. Tham số và tách nhóm test

Nhóm test giữ nguyên theo danh sách chính thức. Phần còn lại (`train_val`) sẽ được chia ở các mục sau.

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax',
    'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural_Thickening', 'Hernia',
]

N_SPLITS = 200      # số cách chia thử
VAL_FRAC = 0.125    # tỉ lệ bệnh nhân làm validation
MIN_POS_VAL = 10    # mỗi bệnh phải có ít nhất chừng này ảnh dương tính trong validation
SEED = 42

test_df = df[df['official'] == 'test'].reset_index(drop=True)
trainval_df = df[df['official'] == 'train_val'].reset_index(drop=True)
print('Test:', len(test_df), 'ảnh | Train_val:', len(trainval_df), 'ảnh')

## 4. Thử 200 cách chia và chọn cách tốt nhất

Với mỗi cách chia: loại nếu có bệnh nào ít hơn 10 ảnh dương tính trong validation. Với các cách còn lại, điểm số là **độ lệch tương đối** `|tỉ lệ train - tỉ lệ val| / tỉ lệ train`, lấy trung bình trên 14 bệnh (dùng độ lệch tương đối để các bệnh hiếm như Hernia không bị lấn át). Chọn cách có điểm thấp nhất.

In [ ]:
labels = trainval_df[DISEASES].to_numpy()
groups = trainval_df['patient'].to_numpy()

gss = GroupShuffleSplit(n_splits=N_SPLITS, test_size=VAL_FRAC, random_state=SEED)

best_score = np.inf
best_split = None
n_valid = 0

for i, (tr_idx, va_idx) in enumerate(gss.split(trainval_df, groups=groups)):
    val_pos = labels[va_idx].sum(axis=0)
    if (val_pos < MIN_POS_VAL).any():
        continue
    n_valid += 1

    p_train = labels[tr_idx].mean(axis=0)
    p_val = labels[va_idx].mean(axis=0)
    score = np.mean(np.abs(p_train - p_val) / p_train)

    if score < best_score:
        best_score = score
        best_split = (i, tr_idx, va_idx)

if best_split is None:
    raise RuntimeError(
        f'Không có cách chia nào thỏa điều kiện >= {MIN_POS_VAL} ảnh dương tính/bệnh '
        f'trong validation. Hãy tăng N_SPLITS hoặc giảm MIN_POS_VAL.'
    )

split_id, tr_idx, va_idx = best_split
print(f'Số cách chia hợp lệ: {n_valid}/{N_SPLITS}')
print(f'Chọn cách chia #{split_id}, độ lệch tương đối trung bình = {best_score:.4f}')

train_df = trainval_df.iloc[tr_idx].reset_index(drop=True)
val_df = trainval_df.iloc[va_idx].reset_index(drop=True)

## 5. Kiểm tra rò rỉ

Khẳng định không có bệnh nhân nào xuất hiện ở hai nhóm khác nhau, và không có ảnh nào nằm ở hai nhóm. Nếu đúng sẽ in `Leakage check passed`.

In [ ]:
sets = {
    'train': set(train_df['patient']),
    'val': set(val_df['patient']),
    'test': set(test_df['patient']),
}
names = list(sets)
for a in range(len(names)):
    for b in range(a + 1, len(names)):
        overlap = sets[names[a]] & sets[names[b]]
        assert len(overlap) == 0, (
            f'Leakage: {len(overlap)} patient trùng giữa {names[a]} và {names[b]}'
        )

assert not (set(train_df['image']) & set(val_df['image']))
assert not (set(train_df['image']) & set(test_df['image']))
assert not (set(val_df['image']) & set(test_df['image']))

print('Leakage check passed')

## 6. Lưu ba file lên Drive

Ô này **không ghi đè** nếu ba file đã tồn tại: nó chỉ kiểm tra file cũ có khớp với cách chia vừa tính hay không. Việc này nhằm tránh làm đổi nhóm train/val sau khi đã huấn luyện (mô hình có thể đã nhìn thấy ảnh nằm trong validation mới, khiến điểm validation đẹp giả). Chỉ đặt `OVERWRITE = True` khi chắc chắn muốn chia lại, và khi đó phải huấn luyện lại từ đầu.

In [ ]:
OVERWRITE = False

new = {'train': train_df, 'val': val_df, 'test': test_df}
paths = {n: OUT_DIR + n + '.csv' for n in new}
existing = [n for n in new if os.path.exists(paths[n])]

if existing and not OVERWRITE:
    if len(existing) < len(new):
        raise RuntimeError(f'Chỉ có một phần file tồn tại ({existing}). Kiểm tra thư mục {OUT_DIR}')
    same = all(set(pd.read_csv(paths[n])['image']) == set(new[n]['image']) for n in new)
    if same:
        print('Ba file đã tồn tại và khớp với cách chia vừa tính: giữ nguyên, không ghi đè.')
    else:
        raise RuntimeError(
            'Ba file đã tồn tại nhưng KHÁC cách chia vừa tính. Không ghi đè để tránh rò rỉ. '
            'Đặt OVERWRITE = True chỉ khi chắc chắn, và phải huấn luyện lại từ đầu.'
        )
else:
    for n in new:
        new[n].to_csv(paths[n], index=False)
    print('Đã lưu train.csv, val.csv, test.csv vào', OUT_DIR)

## 7. Thống kê ba nhóm

Kết quả đã chạy được với `SEED = 42`: 188/200 cách chia hợp lệ, chọn cách số 33 (độ lệch 0,0454); train 75.860 ảnh / 24.507 bệnh nhân, validation 10.664 ảnh / 3.501 bệnh nhân, test 25.596 ảnh / 2.797 bệnh nhân. Con số có thể khác nếu phiên bản scikit-learn hay thứ tự dòng của `meta.csv` khác.

In [ ]:
for name, d in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f"{name}: {len(d)} ảnh, {d['patient'].nunique()} bệnh nhân")

print('\nSố ảnh dương tính từng bệnh trong validation:')
print(val_df[DISEASES].sum().astype(int).to_string())

print('\nTỉ lệ bệnh:')
print(pd.DataFrame({
    'train': train_df[DISEASES].mean(),
    'val': val_df[DISEASES].mean(),
    'test': test_df[DISEASES].mean(),
}).round(4))

## 8. Kiểm tra lại từ file đã lưu

Đọc lại ba file từ Drive (không dùng biến trong bộ nhớ) để chắc chắn thứ được lưu là thứ đã kiểm tra.

In [ ]:
tr, va, te = (pd.read_csv(OUT_DIR + f) for f in ('train.csv', 'val.csv', 'test.csv'))

print('train / val / test:', len(tr), len(va), len(te))
print('train + val:', len(tr) + len(va), '(phải bằng số ảnh train_val:', len(trainval_df), ')')
assert len(tr) + len(va) == len(trainval_df)
assert len(te) == len(test_df)
assert not (set(tr['patient']) & set(va['patient']))
assert not (set(tr['patient']) & set(te['patient']))
assert not (set(va['patient']) & set(te['patient']))
print('Hernia trong val:', int(va['Hernia'].sum()))
print('Kiểm tra từ file đã lưu: đạt')

## 9. Lưu ý khi dùng kết quả chia

- **Không chạy lại cách chia khi đã huấn luyện.** Muốn thử cách chia khác, lưu sang thư mục mới và huấn luyện lại từ đầu.
- **Test có tỉ lệ bệnh cao hơn train và validation** (ví dụ Pneumothorax khoảng 10% so với 3%). Đây là đặc điểm của danh sách test chính thức NIH, không phải lỗi.
- **Hernia trong validation chỉ có 18 ca dương tính**, nên kết quả riêng bệnh này dao động mạnh.
- Ảnh là bản resize 224x224 do một người dùng Kaggle thực hiện, không phải bản gốc NIH.
- Trước khi đưa lên GitHub: vào Edit, Clear all outputs, và kiểm tra không còn thông tin cá nhân trong phần kết quả của các ô.